# 1. GraphRAG Pipeline — Medical Textbook AlzRAGBench

**Part of the AlzRAGBench project** — Medical Textbook variant.

This notebook implements **GraphRAG**: knowledge graph construction, structural analysis, static and interactive visualization, and 1-hop subgraph retrieval for Alzheimer's disease queries.

## 1.1 Imports and Paths

In [ ]:
import os
import sys
import json
import time
from pathlib import Path

import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pyvis.network import Network
from IPython.display import IFrame, display

# Paths
NOTEBOOK_DIR = Path(".").resolve()
BASE_DIR     = NOTEBOOK_DIR.parent
DATASET_DIR  = BASE_DIR / "Dataset"
NODES_CSV    = DATASET_DIR / "Knowledge graph" / "nodes.csv"
EDGES_CSV    = DATASET_DIR / "Knowledge graph" / "edges.csv"
RESULTS_DIR  = BASE_DIR / "Results"
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Nodes CSV exists: {NODES_CSV.exists()}")
print(f"Edges CSV exists: {EDGES_CSV.exists()}")

## 1.2 Load and Build Knowledge Graph with NetworkX

In [ ]:
nodes_df = pd.read_csv(NODES_CSV)
edges_df = pd.read_csv(EDGES_CSV)

G = nx.Graph()

for _, row in nodes_df.iterrows():
    G.add_node(
        row["nodeId:ID"],
        name=row["name"],
        label=row["type:LABEL"],
        description=row["description"]
    )

for _, row in edges_df.iterrows():
    G.add_edge(
        row[":START_ID"],
        row[":END_ID"],
        relation=row[":TYPE"],
        evidence=row["evidence"]
    )

print(f"Graph Statistics:")
print(f"  Total Nodes: {G.number_of_nodes()}")
print(f"  Total Edges: {G.number_of_edges()}")
print(f"  Is Connected: {nx.is_connected(G)}")

## 1.3 Knowledge Graph Visualizations

In [ ]:
# Static Visualization
TYPE_COLORS = {
    "Disease":      "#E05C5C",
    "Drug":         "#5C8BE0",
    "Gene":         "#5CE07A",
    "GeneVariant":  "#B45CE0",
    "Protein":      "#E0A85C",
    "RiskFactor":   "#5CCFE0",
    "Biomarker":    "#E0D85C",
    "Mechanism":    "#E07ABB",
    "Pathology":    "#A0A0A0"
}

node_colors = [TYPE_COLORS.get(G.nodes[n].get("label", ""), "#cccccc") for n in G.nodes()]
node_sizes  = [300 + G.degree(n) * 80 for n in G.nodes()]

fig, ax = plt.subplots(figsize=(15, 10))
pos = nx.spring_layout(G, seed=42, k=2.0)

nx.draw_networkx_edges(G, pos, alpha=0.25, edge_color="#aaaaaa", ax=ax)
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.9, ax=ax)
labels = {n: G.nodes[n].get("name", n) for n in G.nodes()}
nx.draw_networkx_labels(G, pos, labels, font_size=7, ax=ax)

patches = [mpatches.Patch(color=c, label=t) for t, c in TYPE_COLORS.items()]
ax.legend(handles=patches, loc="upper left", title="Entity Types", fontsize=8)
ax.set_title("Alzheimer's Disease Knowledge Graph", fontsize=14, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "knowledge_graph_static.png", dpi=150)
plt.show()
print("Saved static graph visualization.")

In [ ]:
# Interactive Visualization (Pyvis)
net = Network(height="550px", width="100%", bgcolor="#1a1a2e", font_color="white")
net.barnes_hut()

for n in G.nodes():
    data = G.nodes[n]
    c = TYPE_COLORS.get(data.get("label", ""), "#cccccc")
    net.add_node(str(n), label=data.get("name", str(n)), title=f"{data.get('label','')}: {data.get('description','')}", color=c)

for u, v, d in G.edges(data=True):
    net.add_edge(str(u), str(v), title=d.get("relation", ""), color="#666666")

html_out = str(RESULTS_DIR / "knowledge_graph_interactive.html")
net.save_graph(html_out)
display(IFrame(html_out, width="100%", height="570px"))

## 1.4 GraphRAG Retrieval Function

In [ ]:
def graph_retrieve_context(query):
    """Find matching entities and extract 1-hop relation neighborhood."""
    q = query.lower()
    matched_nodes = [
        n for n, d in G.nodes(data=True)
        if q in str(d.get("name", "")).lower()
        or q in str(d.get("description", "")).lower()
    ]
    
    if not matched_nodes:
        return ""
        
    context = ""
    visited = set()
    for node in matched_nodes:
        if node in visited:
            continue
        visited.add(node)
        nd = G.nodes[node]
        context += f"Entity: {nd.get('name', '')}\nType: {nd.get('label', '')}\nDescription: {nd.get('description', '')}\nRelations:\n"
        for nbr in G.neighbors(node):
            e = G.get_edge_data(node, nbr)
            nbr_name = G.nodes[nbr].get("name", nbr)
            context += f"  -- {e.get('relation', '')} --> {nbr_name}\n"
        context += "\n"
    return context

# Test retrieval
test_context = graph_retrieve_context("APOE")
print("Retrieved GraphRAG Context:")
print(test_context[:400] + "...")